# TSA29 Mini-Instance: ws3 Harvest Scenario Demo

This notebook demonstrates running a **basic areacontrol priority-queue heuristic even-flow harvest scenario** on the TSA29 mini-instance ws3 model.

**Model summary:**
- 9,788 fragments, 90,499.8 ha
- 72 development types (combinations of AU × IFM × ORIGIN × SILV_STATE)
- 150 yield curves
- 30-period horizon (300 years), 10-year periods

**Scenario:** Even-flow harvest targeting a constant annual harvest area across all periods, using a priority-queue heuristic that selects the oldest operable age classes first.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

# Add ws3 to path
sys.path.insert(0, str(Path(__file__).parent.parent.parent / 'src'))

import ws3.core
import ws3.forest

INSTANCE_ROOT = Path(__file__).parent.parent
MODEL_DIR = INSTANCE_ROOT / 'models' / 'ws3_model'
FRAGMENTS_PATH = INSTANCE_ROOT / 'output' / 'patchworks_tsa29mini' / 'fragments' / 'fragments.shp'

print(f'Instance root: {INSTANCE_ROOT}')
print(f'Model dir: {MODEL_DIR}')
print(f'Fragments: {FRAGMENTS_PATH}')

In [ ]:
import geopandas as gpd
import xml.etree.ElementTree as ET

# Load fragments for reference
fragments = gpd.read_file(FRAGMENTS_PATH)
print(f'Fragments: {len(fragments)}, Area: {fragments["AREA_HA"].sum():.1f} ha')

# Load XML curves
tree = ET.parse(INSTANCE_ROOT / 'output' / 'patchworks_tsa29mini' / 'forestmodel.xml')
root = tree.getroot()

curve_defs = {}
for curve_elem in root.findall('.//curve'):
    curve_id = curve_elem.get('id')
    points = []
    for point_elem in curve_elem.findall('point'):
        x = int(point_elem.get('x'))
        y = float(point_elem.get('y'))
        points.append((x, y))
    if len(points) >= 2:
        curve_defs[curve_id] = points

print(f'Curves from XML: {len(curve_defs)}')

In [ ]:
# Build ws3 curves
ws3_curves = {}
for curve_id, points in curve_defs.items():
    ws3_curves[curve_id] = ws3.core.Curve(
        label=curve_id, id=curve_id, points=points, type='a'
    )

# Create ForestModel
model = ws3.forest.ForestModel(
    model_name='tsa29mini',
    model_path=str(INSTANCE_ROOT / 'output' / 'patchworks_tsa29mini'),
    base_year=2026,
    horizon=30,
    period_length=10,
    max_age=300,
)

# Register curves
for curve in ws3_curves.values():
    model.register_curve(curve)

# Create development types from fragments
dt_groups = fragments.groupby(['AU', 'IFM', 'ORIGIN', 'SILV_STATE'])
for (au, ifm, origin, silv_state), group in dt_groups:
    key = (au, ifm, origin, silv_state)
    dt = model.create_dtype_fromkey(key)
    age_area = group.groupby('F_AGE')['AREA_HA'].sum()
    for age, area in age_area.items():
        dt._areas[0][int(age)] = area

print(f'Development types: {len(model.dtypes)}')
print(f'Total area: {model.inventory(period=0):.1f} ha')

In [ ]:
# Define harvest action
# Minimum harvest age: 60 years
MIN_HARVEST_AGE = 60

# Set up operability expression: harvest when age >= MIN_HARVEST_AGE
model.oper_expr['harvest'] = {
    ('?',): f'age >= {MIN_HARVEST_AGE}'
}
model.transitions['harvest'] = {
    ('?',): {
        f'age >= {MIN_HARVEST_AGE}': ('harvested',)
    }
}

# Compile actions for all development types
for dtk in model.dtypes:
    model.dtypes[dtk].compile_action('harvest')

print(f'Harvest action defined: min age = {MIN_HARVEST_AGE}')
print(f'Operable in period 0: {sum(1 for dt in model.dtypes.values() if dt.operability.get("harvest", {}).get(0))} DTs')

In [ ]:
# Even-flow harvest scenario
# Target: harvest equal area each period
TOTAL_AREA = model.inventory(period=0)
HORIZON = model.horizon
TARGET_AREA_PER_PERIOD = TOTAL_AREA / HORIZON  # ~3,017 ha/period

print(f'Total area: {TOTAL_AREA:.1f} ha')
print(f'Horizon: {HORIZON} periods')
print(f'Target area per period: {TARGET_AREA_PER_PERIOD:.1f} ha')
print(f'Target annual harvest: {TARGET_AREA_PER_PERIOD / model.period_length:.1f} ha/yr')

# Run the scenario
period_harvest_areas = []
period_remaining_area = []

remaining_area = TOTAL_AREA

for period in range(HORIZON):
    # Calculate target for this period
    periods_left = HORIZON - period
    if periods_left > 0:
        target = remaining_area / periods_left
    else:
        target = 0
    
    # Use the GreedyAreaSelector (oldest first)
    selector = ws3.forest.GreedyAreaSelector(model)
    missing = selector.operate(
        period=period,
        acode='harvest',
        target_area=target,
        commit_actions=True,
    )
    
    harvested = target - missing if missing < target else target
    period_harvest_areas.append(harvested)
    remaining_area -= harvested
    period_remaining_area.append(remaining_area)
    
    if period % 5 == 0:
        print(f'Period {period:2d}: harvested={harvested:8.1f} ha, remaining={remaining_area:8.1f} ha')

print(f'\nFinal remaining area: {remaining_area:.1f} ha')
print(f'Total harvested: {TOTAL_AREA - remaining_area:.1f} ha')

In [ ]:
# Compile results
results = pd.DataFrame({
    'period': list(range(HORIZON)),
    'harvest_area_ha': period_harvest_areas,
    'remaining_area_ha': period_remaining_area,
    'harvest_rate': [h / model.period_length for h in period_harvest_areas],
})

print('Harvest scenario results:')
print(results.to_string(index=False))
print(f'\nMean harvest/period: {results["harvest_area_ha"].mean():.1f} ha')
print(f'Std dev harvest/period: {results["harvest_area_ha"].std():.1f} ha')
print(f'CV: {results["harvest_area_ha"].std() / results["harvest_area_ha"].mean() * 100:.1f}%')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Harvest area per period
ax = axes[0, 0]
ax.bar(results['period'], results['harvest_area_ha'], color='steelblue', alpha=0.8)
ax.axhline(TARGET_AREA_PER_PERIOD, color='red', linestyle='--', label=f'Target ({TARGET_AREA_PER_PERIOD:.0f} ha)')
ax.set_xlabel('Period')
ax.set_ylabel('Harvest Area (ha)')
ax.set_title('Harvest Area by Period')
ax.legend()
ax.set_xlim(-0.5, HORIZON - 0.5)

# Plot 2: Remaining area over time
ax = axes[0, 1]
ax.plot(results['period'], results['remaining_area_ha'], 'o-', color='forestgreen', linewidth=2)
ax.set_xlabel('Period')
ax.set_ylabel('Remaining Area (ha)')
ax.set_title('Remaining Forested Area Over Time')
ax.set_xlim(-0.5, HORIZON - 0.5)

# Plot 3: Harvest rate (ha/yr)
ax = axes[1, 0]
ax.bar(results['period'], results['harvest_rate'], color='darkorange', alpha=0.8)
target_rate = TARGET_AREA_PER_PERIOD / model.period_length
ax.axhline(target_rate, color='red', linestyle='--', label=f'Target ({target_rate:.1f} ha/yr)')
ax.set_xlabel('Period')
ax.set_ylabel('Harvest Rate (ha/yr)')
ax.set_title('Annual Harvest Rate')
ax.legend()
ax.set_xlim(-0.5, HORIZON - 0.5)

# Plot 4: Cumulative harvest
ax = axes[1, 1]
cumulative = results['harvest_area_ha'].cumsum()
ax.plot(results['period'], cumulative, 's-', color='crimson', linewidth=2)
ax.axhline(TOTAL_AREA, color='gray', linestyle=':', label=f'Total area ({TOTAL_AREA:.0f} ha)')
ax.set_xlabel('Period')
ax.set_ylabel('Cumulative Harvest (ha)')
ax.set_title('Cumulative Harvest Over Time')
ax.legend()
ax.set_xlim(-0.5, HORIZON - 0.5)

plt.tight_layout()
plt.savefig(INSTANCE_ROOT / 'output' / 'patchworks_tsa29mini' / 'ws3_model' / 'harvest_scenario_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: harvest_scenario_results.png')

In [ ]:
# Development type breakdown
dt_summary = []
for key, dt in model.dtypes.items():
    area_0 = dt._areas[0]
    if not area_0:
        continue
    min_age = min(area_0.keys())
    max_age = max(area_0.keys())
    mean_age = sum(a * a_ha for a, a_ha in area_0.items()) / sum(area_0.values())
    operable_0 = dt.operable_area('harvest', 0)
    dt_summary.append({
        'AU': key[0],
        'IFM': key[1],
        'ORIGIN': key[2],
        'SILV_STATE': key[3],
        'Total_Area_ha': sum(area_0.values()),
        'Min_Age': min_age,
        'Max_Age': max_age,
        'Mean_Age': round(mean_age, 1),
        'Operable_Area_p0_ha': operable_0,
    })

dt_df = pd.DataFrame(dt_summary)
print(f'Development types: {len(dt_df)}')
print(f'\nTotal area by IFM:')
print(dt_df.groupby('IFM')['Total_Area_ha'].sum().to_string())
print(f'\nTotal area by ORIGIN:')
print(dt_df.groupby('ORIGIN')['Total_Area_ha'].sum().to_string())
print(f'\nOperable area in period 0: {dt_df["Operable_Area_p0_ha"].sum():.1f} ha')
print(f'\nFirst 10 DTs:')
print(dt_df.head(10).to_string(index=False))

## Summary

This demo built a ws3 ForestModel from the TSA29 mini-instance Patchworks output and ran a basic even-flow harvest scenario:

1. **Loaded** 9,788 fragments (90,499.8 ha) and 152 yield curves from the Patchworks export
2. **Built** 72 development types representing all AU × IFM × ORIGIN × SILV_STATE combinations
3. **Defined** a harvest action with minimum age 60 years
4. **Ran** a 30-period even-flow scenario targeting ~3,017 ha/period using the GreedyAreaSelector (oldest-first priority queue)
5. **Visualized** harvest area, remaining area, harvest rate, and cumulative harvest

The even-flow target is approximately met, with the priority-queue heuristic selecting oldest operable stands first each period.